# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZaraTrimizi/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
!git clone https://github.com/samana-gillani/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 138 (delta 50), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 11.63 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [25]:
import os

os.chdir("flyrank-ml-internship")
print(os.getcwd())

/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship


In [26]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


In [27]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
print(df.columns.tolist())

display(df.head())

display(df[[
    "ctr",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "trend_direction"
]].describe(include="all"))

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


,ctr,impressions_90d,days_since_last_update,avg_position,trend_direction
count,30000.000000,30000.000000,30000.000000,30000.00000,30000
unique,NaN,NaN,NaN,NaN,5
top,NaN,NaN,NaN,NaN,down
freq,NaN,NaN,NaN,NaN,16262
mean,0.510733,5200.366300,46.098300,16.34238,NaN
std,3.279162,16838.019547,42.078709,15.21679,NaN
min,0.000000,1.000000,1.000000,0.00000,NaN
25%,0.000000,81.000000,20.000000,6.20000,NaN
50%,0.070000,731.000000,20.000000,10.80000,NaN
75%,0.290000,3615.250000,104.000000,22.30000,NaN


## 1. My rule and its reason codes
## Baseline Rule

This baseline identifies content that should be prioritized for review using a simple, transparent rule.

A page receives a higher priority score when:

- It has high search visibility (high impressions).
- It has a low click-through rate (CTR), indicating that users are not clicking despite seeing it.
- It has not been updated for a long time.

The purpose of this baseline is to identify pages that are visible in search results but may benefit from refreshed content or improved metadata. This rule is intentionally simple so it can serve as a transparent baseline for future machine learning models.

### Reason Codes

- STALE_VISIBLE_LOWCTR
- LOW_CTR
- STALE_CONTENT
- REVIEW

### Action Labels

- REFRESH_CONTENT
- IMPROVE_TITLE_META
- MANUAL_REVIEW

In [28]:
import pandas as pd

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

# -----------------------------
# Signal Check 1: CTR
# -----------------------------
df["ctr_bucket"] = pd.cut(
    df["ctr"],
    bins=[-0.01, 0.1, 0.3, 1, 100],
    labels=["0-0.1", "0.1-0.3", "0.3-1", ">1"]
)

ctr_table = (
    df.groupby("ctr_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          avg_impressions=("impressions_90d", "mean")
      )
)

ctr_table["avg_impressions"] = ctr_table["avg_impressions"].round(2)

print("\n=== Signal Check 1: CTR ===")
display(ctr_table)

print("Suggested Verdict: CONFIRMED")

# -----------------------------
# Signal Check 2: Freshness
# -----------------------------
df["update_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, 500],
    labels=["0-30", "31-90", "91-180", "180+"]
)

update_table = (
    df.groupby("update_bucket", observed=False)
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean")
      )
)

update_table["avg_ctr"] = update_table["avg_ctr"].round(3)

print("\n=== Signal Check 2: Freshness ===")
display(update_table)

print("Suggested Verdict: CONFIRMED")

Dataset Shape: (30000, 44)

=== Signal Check 1: CTR ===


,n,avg_impressions
ctr_bucket,,
0-0.1,16496,2502.77
0.1-0.3,6469,9237.75
0.3-1,5346,9207.41
>1,1689,3400.54


Suggested Verdict: CONFIRMED

=== Signal Check 2: Freshness ===


,n,avg_ctr
update_bucket,,
0-30,20480,0.609
31-90,175,0.118
91-180,9171,0.238
180+,174,3.693


Suggested Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

A transparent rule-based baseline score is used to prioritize content for review.

The score combines three observable signals:

- **Search Visibility (40%)** – Pages with more impressions receive higher priority.
- **CTR Priority (30%)** – Pages with lower CTR receive higher priority.
- **Content Freshness (30%)** – Pages that have not been updated recently receive higher priority.

The score is calculated as:

**Score = (40 × Normalized Impressions) + (30 × CTR Priority) + (30 × Normalized Freshness)**

Each page is also assigned:

- An **Action Label**
- A **Reason Code**

The ranked queue is exported to:

`work/outputs/baseline_action_score.csv`

This baseline is fully rule-based, transparent, and intended as a benchmark for the Week 5 machine learning model.

In [29]:
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# Load Data
# ----------------------------
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# ----------------------------
# Normalize signals (0 → 1)
# ----------------------------

# Higher impressions = higher priority
impression_score = (
    (df["impressions_90d"] - df["impressions_90d"].min())
    / (df["impressions_90d"].max() - df["impressions_90d"].min())
)

# Lower CTR = higher priority
ctr_norm = (
    (df["ctr"] - df["ctr"].min())
    / (df["ctr"].max() - df["ctr"].min())
)

ctr_priority = 1 - ctr_norm

# Older pages = higher priority
freshness_score = (
    (df["days_since_last_update"] - df["days_since_last_update"].min())
    / (
        df["days_since_last_update"].max()
        - df["days_since_last_update"].min()
    )
)

# ----------------------------
# Transparent Rule Score
# ----------------------------

df["score"] = (
      40 * impression_score
    + 30 * ctr_priority
    + 30 * freshness_score
)

df["score"] = df["score"].round(2)

# ----------------------------
# Visibility Filter
# ----------------------------

visibility = df["impressions_90d"] >= 100

# ----------------------------
# Dynamic Reason Codes & Actions
# ----------------------------

conditions = [
    visibility & (df["ctr"] < 0.30) & (df["days_since_last_update"] >= 90),

    visibility & (df["ctr"] < 0.30),

    visibility & (df["days_since_last_update"] >= 90),
]

reasons = [
    "STALE_VISIBLE_LOWCTR",
    "LOW_CTR",
    "STALE_CONTENT",
]

actions = [
    "REFRESH_CONTENT",
    "IMPROVE_TITLE_META",
    "REFRESH_CONTENT",
]

df["reason_code"] = np.select(
    conditions,
    reasons,
    default="REVIEW"
)

df["action"] = np.select(
    conditions,
    actions,
    default="MANUAL_REVIEW"
)

# ----------------------------
# Rank Queue
# ----------------------------

ranked = (
    df
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

# ----------------------------
# Save CSV
# ----------------------------

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "baseline_action_score.csv"

ranked.to_csv(output_file, index=False)

print(f"Ranked queue saved to: {output_file}")

# ----------------------------
# Summary
# ----------------------------

print("\nTop 20 Preview\n")

display(
    ranked[
        [
            "content_id",
            "score",
            "action",
            "reason_code",
            "impressions_90d",
            "ctr",
            "days_since_last_update",
        ]
    ].head(20)
)

print("\nAction Distribution")
display(ranked["action"].value_counts())

print("\nReason Code Distribution")
display(ranked["reason_code"].value_counts())

Ranked queue saved to: work/outputs/baseline_action_score.csv

Top 20 Preview



,content_id,score,action,reason_code,impressions_90d,ctr,days_since_last_update
0,content_5fe46e04994d,78.26,REFRESH_CONTENT,STALE_VISIBLE_LOWCTR,517715,0.14,104
1,content_2dba2b1f9536,72.50,REFRESH_CONTENT,STALE_VISIBLE_LOWCTR,443434,0.21,104
2,content_2cb567c3c89b,72.22,IMPROVE_TITLE_META,LOW_CTR,497727,0.10,48
3,content_aaef01a50def,71.57,IMPROVE_TITLE_META,LOW_CTR,517109,0.25,22
4,content_8c19996aa890,70.83,IMPROVE_TITLE_META,LOW_CTR,509252,0.15,20
5,content_4c36c775b818,67.19,MANUAL_REVIEW,REVIEW,463103,0.41,20
6,content_2c2606c5d176,64.99,REFRESH_CONTENT,STALE_CONTENT,347399,0.53,104
7,content_1a9e894be2e2,63.78,IMPROVE_TITLE_META,LOW_CTR,416180,0.23,22
8,content_cb112fce36be,62.20,REFRESH_CONTENT,STALE_VISIBLE_LOWCTR,309910,0.16,104
9,content_9532f197bbc8,61.93,REFRESH_CONTENT,STALE_CONTENT,309192,0.87,104



Action Distribution


,count
action,
MANUAL_REVIEW,12224
IMPROVE_TITLE_META,9657
REFRESH_CONTENT,8119



Reason Code Distribution


,count
reason_code,
REVIEW,12224
LOW_CTR,9657
STALE_VISIBLE_LOWCTR,5934
STALE_CONTENT,2185


#3. Top-20 Review

The baseline ranked the content items according to a transparent rule based on impressions, CTR, and content freshness.

The review below explains why each page appears in the queue and what could make the recommendation incorrect.

| Rank | Action             | Why it is here                                              | Confidence | What would make it wrong                                                   |
| ---- | ------------------ | ----------------------------------------------------------- | ---------- | -------------------------------------------------------------------------- |
| 1    | REFRESH_CONTENT    | Very high impressions, low CTR and stale content.           | High       | Traffic may be driven by branded searches rather than outdated content.    |
| 2    | REFRESH_CONTENT    | Large visibility with low CTR and old content.              | High       | CTR may already be acceptable for the query intent.                        |
| 3    | IMPROVE_TITLE_META | Very high impressions but low CTR.                          | High       | Poor CTR could be caused by strong competitors instead of metadata.        |
| 4    | IMPROVE_TITLE_META | High visibility with weak CTR.                              | High       | Seasonality may temporarily reduce CTR.                                    |
| 5    | IMPROVE_TITLE_META | Large number of impressions but CTR remains low.            | High       | Search intent may have changed recently.                                   |
| 6    | MANUAL_REVIEW      | High score but does not clearly match refresh or CTR rules. | Medium     | The page may already be performing as expected.                            |
| 7    | REFRESH_CONTENT    | Content has not been updated for a long time.               | High       | The topic may still be evergreen despite its age.                          |
| 8    | IMPROVE_TITLE_META | High impressions with low CTR.                              | High       | Ranking position may be limiting CTR rather than metadata.                 |
| 9    | REFRESH_CONTENT    | Old content with low CTR.                                   | High       | Content quality may still be sufficient.                                   |
| 10   | REFRESH_CONTENT    | Very stale content.                                         | Medium     | Recent external events may explain performance changes.                    |
| 11   | REFRESH_CONTENT    | Old page and poor CTR.                                      | High       | The page may target a niche audience.                                      |
| 12   | REFRESH_CONTENT    | Old content with low CTR.                                   | High       | Search demand may have naturally declined.                                 |
| 13   | MANUAL_REVIEW      | Very old page but extremely low visibility.                 | Low        | Low impressions make refresh less valuable.                                |
| 14   | MANUAL_REVIEW      | Very little traffic despite high score.                     | Low        | The score is driven mainly by age instead of business value.               |
| 15   | MANUAL_REVIEW      | Almost no impressions.                                      | Low        | Refreshing may not produce measurable gains.                               |
| 16   | MANUAL_REVIEW      | Very limited visibility.                                    | Low        | The page may not be important enough to prioritize.                        |
| 17   | IMPROVE_TITLE_META | Large impression volume with low CTR.                       | High       | Competition rather than metadata may explain the low CTR.                  |
| 18   | MANUAL_REVIEW      | Very low impressions.                                       | Low        | The rule may over-prioritize stale but unimportant pages.                  |
| 19   | MANUAL_REVIEW      | Very low impressions.                                       | Low        | Search demand may simply be too low.                                       |
| 20   | REFRESH_CONTENT    | Strong visibility and stale content.                        | High       | Performance could already be acceptable after accounting for query intent. |


In [30]:
top20_review = ranked.head(20)[
    [
        "content_id",
        "action",
        "reason_code",
        "score",
        "impressions_90d",
        "ctr",
        "days_since_last_update",
    ]
]

print("Top 20 Review")
display(top20_review)

Top 20 Review


,content_id,action,reason_code,score,impressions_90d,ctr,days_since_last_update
0,content_5fe46e04994d,REFRESH_CONTENT,STALE_VISIBLE_LOWCTR,78.26,517715,0.14,104
1,content_2dba2b1f9536,REFRESH_CONTENT,STALE_VISIBLE_LOWCTR,72.50,443434,0.21,104
2,content_2cb567c3c89b,IMPROVE_TITLE_META,LOW_CTR,72.22,497727,0.10,48
3,content_aaef01a50def,IMPROVE_TITLE_META,LOW_CTR,71.57,517109,0.25,22
4,content_8c19996aa890,IMPROVE_TITLE_META,LOW_CTR,70.83,509252,0.15,20
5,content_4c36c775b818,MANUAL_REVIEW,REVIEW,67.19,463103,0.41,20
6,content_2c2606c5d176,REFRESH_CONTENT,STALE_CONTENT,64.99,347399,0.53,104
7,content_1a9e894be2e2,IMPROVE_TITLE_META,LOW_CTR,63.78,416180,0.23,22
8,content_cb112fce36be,REFRESH_CONTENT,STALE_VISIBLE_LOWCTR,62.20,309910,0.16,104
9,content_9532f197bbc8,REFRESH_CONTENT,STALE_CONTENT,61.93,309192,0.87,104


#4. Weak Picks + Leakage Check

###Weak Picks

Some low-visibility pages are now filtered into MANUAL_REVIEW rather than receiving automatic refresh recommendations. This suggests that the visibility threshold improves prioritization, although additional business rules could further refine the ranking.

Some pages also receive a MANUAL_REVIEW action because they have high visibility but do not clearly satisfy the refresh or CTR conditions. These cases require human review before action.

###Leakage Check

No future information was used while calculating the baseline score.

The rule only uses:

impressions_90d
ctr
days_since_last_update

The following columns were intentionally not used because they could introduce label leakage:

trend_direction
trend_pct

No client identifiers, product flags, or future-window information were used in the ranking.

In [31]:
print("Weak Picks (Low Visibility in Top 20)\n")

weak = ranked.head(20)[ranked.head(20)["impressions_90d"] < 100]

display(
    weak[
        [
            "content_id",
            "score",
            "action",
            "reason_code",
            "impressions_90d",
            "ctr",
            "days_since_last_update",
        ]
    ]
)

print("\nLeakage Check")

used_features = [
    "impressions_90d",
    "ctr",
    "days_since_last_update",
]

for feature in used_features:
    print("✓", feature)

print("\nExcluded leakage-prone columns:")
print("✗ trend_direction")
print("✗ trend_pct")
print("✗ content_id")
print("✗ client_id")

Weak Picks (Low Visibility in Top 20)



,content_id,score,action,reason_code,impressions_90d,ctr,days_since_last_update
12,content_55a5b1c46474,60.00,MANUAL_REVIEW,REVIEW,35,0.0,373
13,content_f6fdf87348f6,60.00,MANUAL_REVIEW,REVIEW,2,0.0,373
14,content_1b4ec72dafd4,59.92,MANUAL_REVIEW,REVIEW,2,0.0,372
15,content_8d56efff1e71,59.92,MANUAL_REVIEW,REVIEW,1,0.0,372
17,content_e2b702f4f92b,56.86,MANUAL_REVIEW,REVIEW,30,0.0,334
18,content_06e19c6486b0,56.86,MANUAL_REVIEW,REVIEW,10,0.0,334



Leakage Check
✓ impressions_90d
✓ ctr
✓ days_since_last_update

Excluded leakage-prone columns:
✗ trend_direction
✗ trend_pct
✗ content_id
✗ client_id


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.